In [2]:
import anthropic
import os

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_anthropic import ChatAnthropic
from langchain_voyageai import VoyageAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


In [3]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
dataset = client.read_dataset(
        dataset_name="rag-evaluation-dataset",
)

In [4]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('5e6723f8-c952-4495-8040-1f2cfe86c9cc'), created_at=datetime.datetime(2026, 5, 6, 6, 13, 58, 668318, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 6, 6, 13, 58, 668318, tzinfo=TzInfo(0)), example_count=33, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-14.4-arm64-arm-64bit-Mach-O', 'sdk_version': '0.8.0', 'runtime_version': '3.14.3', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': '1.3.2'}})

In [ ]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[1].outputs

{'ground_truth': 'The Iaret smart watch includes 24/7 heart rate monitoring, blood pressure detection, blood oxygen monitoring (not for medical use), and automatic sleep tracking that monitors deep sleep, light sleep, and awake patterns. It features female physiological cycle tracking, drink reminders, and sedentary reminders to help manage lifestyle. The watch supports 8 sports modes (walking, running, cycling, skipping, badminton, basketball, football, swimming) and provides 3-7 days battery life on a single 2-hour charge.',
 'reference_context_ids': ['B0BCPZ2D8G'],
 'reference_descriptions': ['Iaret Smart Watch for Women(Call Receive/Dial), Fitness Tracker Waterproof Smartwatch for Android iOS Phones 1.7" HD Full Touch Screen Digital Watches with Heart Rate Sleep Monitor Pedometer, White CALL RECEIVING/DIALING: After connecting to Bluetooth, you can directly make calls and answer calls on your watch. Iaret smart watch with text and call supports quick communication, information remi

In [ ]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[1].inputs

{'question': 'What health monitoring features does the Iaret smart watch offer?'}

In [55]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[8].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[8].outputs

In [56]:
reference_input

{'question': 'What outdoor projection capabilities does the Hivvtui S8 projector offer?'}

In [57]:
reference_output

{'ground_truth': 'The Hivvtui S8 projector delivers 4K HD resolution with 20,000:1 contrast ratio, supports 30-200 inch projection display with 2.6-14.5 feet projection distance, offers 60-100% zoom, features 4P/4D horizontal and vertical corrections, and measures only 4.9×5.5×6.3 inches for portability. It includes 5G WiFi and Bluetooth 5.1 connectivity, built-in 3W dual stereo speakers, and supports multiple device connections including Fire TV, Roku, PS5, and smartphones.',
 'reference_context_ids': ['B0CGDGVFMM'],
 'reference_descriptions': ['Projector with WiFi and Bluetooth, 5G WiFi 4K HD 20000L Portable Movie Projector with Mini Tripod, Outdoor Projector Home Video Smart Projectors Compatible with iOS/Android/Laptop/TV Stick/HDMI/USB 🌟【4K HD Resolution】The Hivvtui S8 movie projector delivers vivid, crisp HD images at a 20,000:1 contrast ratio. To make it perfect for watching movies, and TV, displaying photos, and other decorations, the projector includes a 3LCD color calibrated 

### RAG Pipeline

In [58]:
from dotenv import load_dotenv
import os
import voyageai

from qdrant_client import QdrantClient
import anthropic

def get_embedding(voyageai_client, text, model = 'voyage-3'):
        result = voyageai_client.embed(
                [text],
                model=model,
                input_type="document"
        )
        return result.embeddings[0]


def retrieve_data(voyageai_client, query, qdrant_client, k=5):
        query_embedding = get_embedding(voyageai_client, query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-00",
                query=query_embedding,
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }


def process_context(context):
        formatted_context = ""
        for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
                formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        return formatted_context


def build_pompt(preprocessed_context, question):
        prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products

Context:
{preprocessed_context}

Question:
{question}
        """
        return prompt


def generate_answer(anthropic_client, prompt):
        message = anthropic_client.messages.create(
                max_tokens=2000,
                messages=[
                        {
                        "role": "user",
                        "content": prompt,
                        }
                ],
                model="claude-haiku-4-5",
        )
        return message.content[0].text


def rag_pipeline(question, top_k=10):
        load_dotenv()
        VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
        voyageai_client = voyageai.Client(api_key=VOYAGE_API_KEY)
        anthropic_client = anthropic.Anthropic()
        qdrant_client = QdrantClient(url='http://localhost:6333')
        retrieved_context = retrieve_data(voyageai_client, question, qdrant_client, top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_pompt(preprocessed_context, question)
        answer = generate_answer(anthropic_client, prompt)

        # for evaluation we should return the following
        final_result = {
                "answer": answer,
                "question": question,
                "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
                "retrieved_context": retrieved_context["retrieved_context"],
                "similarity_scores": retrieved_context["similarity_scores"]
        }

        return final_result

In [16]:
rag_pipeline("Can I get some charger", top_k=5)

{'answer': 'Yes! Based on the available products, I have several charger options for you:\n\n1. **iPhone Fast Charger** (ID: B0C6KVXZH9) - Rating: 4.5\n   - 20W PD Type C Power Wall Charger with 3FT Lightning Cable\n   - Apple MFi Certified\n   - Compatible with iPhone 13/12/11 and other Apple devices\n   - 4X faster than regular chargers\n\n2. **iPhone Charger Cable - 3 Pack** (ID: B09NCXYHMV) - Rating: 4.4\n   - USB-A to Lightning Cord (3.3FT)\n   - Apple MFi Certified\n   - Supports fast charging (2.4A) and data transfer\n   - Compatible with iPhone 14/13/12/11 and other Apple devices\n\n3. **USB Type C to 3.5mm Headphone and Charger Adapter** (ID: B0BCKCJQPN) - Rating: 4.4\n   - 2-in-1 adapter for charging and audio\n   - Supports up to 60W PD fast charging\n   - Compatible with Samsung, iPad Pro, MacBook, Pixel and more\n\nWould you like more details about any of these chargers?',
 'question': 'Can I get some charger',
 'retrieved_context_ids': ['B0B4WNFTVZ',
  'B09F36P17Y',
  'B0

### RAGAS metrics

In [17]:
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
from ragas.dataset_schema import SingleTurnSample

/var/folders/wz/3xwnqvgn3xx_qjtzy1_qtgq40000gn/T/ipykernel_26200/3807497385.py:1: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/wz/3xwnqvgn3xx_qjtzy1_qtgq40000gn/T/ipykernel_26200/3807497385.py:1: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/wz/3xwnqvgn3xx_qjtzy1_qtgq40000gn/T/ipykernel_26200/3807497385.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [18]:
ragas_llm = LangchainLLMWrapper(ChatAnthropic(model="claude-haiku-4-5-20251001"))
ragas_embeddings = LangchainEmbeddingsWrapper(VoyageAIEmbeddings(model="voyage-3"))

/var/folders/wz/3xwnqvgn3xx_qjtzy1_qtgq40000gn/T/ipykernel_26200/4284526507.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatAnthropic(model="claude-haiku-4-5-20251001"))
/var/folders/wz/3xwnqvgn3xx_qjtzy1_qtgq40000gn/T/ipykernel_26200/4284526507.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(VoyageAIEmbeddings(model="voyage-3"))


In [63]:
result = rag_pipeline(reference_input["question"], top_k=5)
result

{'answer': '# Hivvtui S8 Projector - Outdoor Projection Capabilities\n\nBased on the available products, the Hivvtui S8 projector offers the following outdoor projection features:\n\n## Screen Size & Projection Distance\n- **30-200 inch projection display** - Provides a giant screen option for outdoor viewing\n- **Projection distance:** 2.6-14.5 feet - Flexible setup options depending on your space\n\n## Display Options\n- **Aspect ratios:** 4:3 and 16:9 - Compatible with various content formats\n- **Zoom capability:** 60-100% - Allows you to adjust the image size\n- **4P/4D horizontal and vertical corrections** - Ensures optimal viewing angles even on uneven outdoor surfaces\n\n## Portability\n- **Compact size:** 4.9 x 5.5 x 6.3 inches - Makes it convenient to carry to outdoor locations\n- Includes a mini tripod for setup\n\n## Image Quality\n- 4K HD resolution with 20,000:1 contrast ratio\n- 20000L brightness technology for outdoor visibility\n- 3LCD color-calibrated display with eye

In [64]:
async def ragas_faithfulness(run, example):
        """
                run -> trace of our RAG pipeline
                example -> 
        """
        sample = SingleTurnSample(
                user_input=run["question"],
                response=run["answer"],
                retrieved_contexts=run["retrieved_context"],
        )
        scorer = Faithfulness(llm=ragas_llm)
        return await scorer.single_turn_ascore(sample)

async def ragas_response_relevancy(run, example):
        """
                run -> trace of our RAG pipeline
                example -> 
        """
        sample = SingleTurnSample(
                user_input=run["question"],
                response=run["answer"],
                retrieved_contexts=run["retrieved_context"],
        )
        scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
        return await scorer.single_turn_ascore(sample)


async def ragas_context_precision_id_based(run, example):
        """
                run -> trace of our RAG pipeline
                example -> 
        """
        print(run["retrieved_context_ids"],example["reference_context_ids"])
        sample = SingleTurnSample(
                retrieved_context_ids=run["retrieved_context_ids"],
                reference_context_ids=example["reference_context_ids"]
        )
        scorer = IDBasedContextPrecision()
        return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_id_based(run, example):
        """
                run -> trace of our RAG pipeline
                example -> 
        """
        print(run["retrieved_context_ids"],example["reference_context_ids"])
        sample = SingleTurnSample(
                retrieved_context_ids=run["retrieved_context_ids"],
                reference_context_ids=example["reference_context_ids"]
        )
        scorer = IDBasedContextRecall()
        return await scorer.single_turn_ascore(sample)

In [65]:
print(await ragas_faithfulness(result, ""))
print(await ragas_response_relevancy(result, ""))

0.75
0.6881313958165696


In [66]:
reference_output

{'ground_truth': 'The Hivvtui S8 projector delivers 4K HD resolution with 20,000:1 contrast ratio, supports 30-200 inch projection display with 2.6-14.5 feet projection distance, offers 60-100% zoom, features 4P/4D horizontal and vertical corrections, and measures only 4.9×5.5×6.3 inches for portability. It includes 5G WiFi and Bluetooth 5.1 connectivity, built-in 3W dual stereo speakers, and supports multiple device connections including Fire TV, Roku, PS5, and smartphones.',
 'reference_context_ids': ['B0CGDGVFMM'],
 'reference_descriptions': ['Projector with WiFi and Bluetooth, 5G WiFi 4K HD 20000L Portable Movie Projector with Mini Tripod, Outdoor Projector Home Video Smart Projectors Compatible with iOS/Android/Laptop/TV Stick/HDMI/USB 🌟【4K HD Resolution】The Hivvtui S8 movie projector delivers vivid, crisp HD images at a 20,000:1 contrast ratio. To make it perfect for watching movies, and TV, displaying photos, and other decorations, the projector includes a 3LCD color calibrated 

In [67]:
print(await ragas_context_precision_id_based(result, reference_output))
print(await ragas_context_recall_id_based(result, reference_output))

['B0CGDGVFMM', 'B0B4WNFTVZ', 'B09F36P17Y', 'B09ZWNVYV2', 'B098RGV2KM'] ['B0CGDGVFMM']
0.2
['B0CGDGVFMM', 'B0B4WNFTVZ', 'B09F36P17Y', 'B09ZWNVYV2', 'B098RGV2KM'] ['B0CGDGVFMM']
1.0
